# Project 5

In [ ]:
%matplotlib inline
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Color-based features

In [ ]:
def color_hist(img, nbins=32, bins_range=(0, 256), convert=None):
    if convert is not None:
        img = cv2.cvtColor(img, convert)
    c1hist, c2hist, c3hist = map(
        lambda i: np.histogram(img[:,:,i], bins=nbins, range=bins_range),
        [0,1,2]

    )
    feature_vec = np.concatenate((c1hist[0], c2hist[0], c3hist[0]))
    return c1hist[0], c2hist[0], c3hist[0], feature_vec


img = plt.imread('color_spaces_examples/000275.png')

hist_features = color_hist(img)
hist_features_HSV = color_hist(img, cv2.COLOR_RGB2HSV)
hist_features_LUV = color_hist(img, cv2.COLOR_RGB2LUV)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import glob


def plot3d(pixels, colors_rgb, axis_labels=list("RGB"),
           axis_limits=((0, 255), (0, 255), (0, 255))):
    """Plot pixels in 3D."""

    # Create figure and 3D axes
    fig = plt.figure(figsize=(8, 8))
    ax = Axes3D(fig)

    # Set axis limits
    ax.set_xlim(*axis_limits[0])
    ax.set_ylim(*axis_limits[1])
    ax.set_zlim(*axis_limits[2])

    # Set axis labels and sizes
    ax.tick_params(axis='both', which='major', labelsize=14, pad=8)
    ax.set_xlabel(axis_labels[0], fontsize=16, labelpad=16)
    ax.set_ylabel(axis_labels[1], fontsize=16, labelpad=16)
    ax.set_zlabel(axis_labels[2], fontsize=16, labelpad=16)

    # Plot pixel values with colors given in colors_rgb
    ax.scatter(
        pixels[:, :, 0].ravel(),
        pixels[:, :, 1].ravel(),
        pixels[:, :, 2].ravel(),
        c=colors_rgb.reshape((-1, 3)),
        edgecolors='none'
    )

    return ax  # return Axes3D object for further manipulation


for filename in sorted(glob.glob('color_spaces_examples/*png')):
    img = cv2.imread(filename)

    # Select a small fraction of pixels to plot by subsampling it
    scale = max(img.shape[0], img.shape[1], 64) / 64  # at most 64 rows and columns
    img_small = cv2.resize(
        img,
        (np.int(img.shape[1] / scale), np.int(img.shape[0] / scale)),
        interpolation=cv2.INTER_NEAREST
    )

    # Convert subsampled image to desired color space(s)
    img_small_RGB = cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB)  # OpenCV uses BGR, matplotlib likes RGB
    img_small_HSV = cv2.cvtColor(img_small, cv2.COLOR_BGR2HSV)
    img_small_LUV = cv2.cvtColor(img_small, cv2.COLOR_BGR2LUV)
    img_small_rgb = img_small_RGB / 255.  # scaled to [0, 1], only for plotting

    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    # Plot and show
    plot3d(img_small_RGB, img_small_rgb)
    plt.show()

    plot3d(img_small_HSV, img_small_rgb, axis_labels=list("HSV"))
    plt.show()
    
    plot3d(img_small_LUV, img_small_rgb, axis_labels=list("LUV"))
    plt.show()

# Image rescaling

In [ ]:
def convert(img, color_space):
    if color_space != 'RGB':
        conversion = {
            'HSV': cv2.COLOR_RGB2HSV,
            'LUV': cv2.COLOR_RGB2LUV
        }[color_space]
        img = cv2.cvtColor(img, conversion)
    return img


def bin_spatial(img, color_space='RGB', size=(32, 32)):
    img = convert(img, color_space)
    img_small = cv2.resize(img, size)
    features = img_small.ravel()
    return features


size = (8, 8)
img = plt.imread('color_spaces_examples/25.png')
feature_vec = bin_spatial(img, color_space='RGB', size=size)

plt.plot(feature_vec)
plt.title('Spatially Binned Features (RGB)')
plt.show()

feature_vec = bin_spatial(img, color_space='HSV', size=size)

plt.plot(feature_vec)
plt.title('Spatially Binned Features (HSV)')
plt.show()

feature_vec = bin_spatial(img, color_space='LUV', size=size)

plt.plot(feature_vec)
plt.title('Spatially Binned Features (LUB)')
plt.show()

# HOG feature extraction

In [ ]:
from skimage.color import rgb2gray
from skimage.feature import hog


def run_hog(img, orient, pix_per_cell, cell_per_block, vis, feature_vec):
    return hog(
        img,
        orientations=orient,
        pixels_per_cell=(pix_per_cell, pix_per_cell), 
        cells_per_block=(cell_per_block, cell_per_block), 
        visualise=vis,
        feature_vector=feature_vec,
        block_norm="L2-Hys"
    )


def get_hog_features(img, orient, pix_per_cell, cell_per_block,
                     channel='ALL', color_space='RGB',  vis=False, feature_vec=True):
    img = convert(img, color_space)
    
    if channel=='ALL':
        hog_features = []
        for c in range(img.shape[2]):
            hog_features.append(
                run_hog(img[:,:,c], orient, pix_per_cell, cell_per_block, vis, feature_vec)
            )
        hog_features = np.ravel(hog_features)        
    else:
        hog_features = run_hog(img[:,:,channel], orient, pix_per_cell, cell_per_block, vis, feature_vec)
        
    return hog_features
    
    
orient = 8
pix_per_cell = 16
cell_per_block = 1

img = plt.imread('color_spaces_examples/25.png')
feature, hog_image = run_hog(img[:,:,0], orient, pix_per_cell, cell_per_block, vis=True, feature_vec=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
ax1.imshow(img)
ax1.set_title('Original image (car)', fontsize=30)
ax2.imshow(hog_image, cmap='gray')
ax2.set_title('HOG features', fontsize=30)

plt.show()
fig.savefig('output_images/car_hog_and_not.png')


img = plt.imread('color_spaces_examples/2.png')
feature, hog_image = run_hog(img[:,:,0], orient, pix_per_cell, cell_per_block, vis=True, feature_vec=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
ax1.imshow(img)
ax1.set_title('Original image (not car)', fontsize=30)
ax2.imshow(hog_image, cmap='gray')
ax2.set_title('HOG features', fontsize=30)

plt.show()
fig.savefig('output_images/not_car_hog_and_not.png')

# Combining features and fitting a model

In [ ]:
import glob


vehicle_filenames = sorted(glob.glob('data/vehicles/*/*png'))
nonvehicle_filenames = sorted(glob.glob('data/non-vehicles/*/*png'))

In [ ]:
from sklearn.model_selection import train_test_split


def extract_features(feature_extractors, img):
    feature_list = []
    for feat_extr in feature_extractors:
        feature_vector = feat_extr(img)
        feature_list.append(feature_vector)
    return np.hstack(feature_list)


def create_dataset(feature_extractors, directories, split=0.8):
    X_train_out, X_test_out = [], []
    y_train_out, y_test_out = [], []
    for directory in directories:
        X_train, X_test, y_train, y_test = get_train_test(feature_extractors, directory, split)
        X_train_out.append(X_train)
        X_test_out.append(X_test)
        y_train_out.append(y_train)
        y_test_out.append(y_test)
    X_train, X_test = map(np.vstack, [X_train_out, X_test_out])
    y_train, y_test = map(np.hstack, [y_train_out, y_test_out])
    return X_train, X_test, y_train, y_test


def extract_X(feature_extractors, filenames):
    out = []
    for filename in filenames:
        img = plt.imread(filename)
        feature_vec = extract_features(feature_extractors, img)
        out.append(feature_vec)
    return np.vstack(out)
    

def get_train_test(feature_extractors, directory, split):
    filenames = sorted(glob.glob(directory + '/*'))
    X = extract_X(feature_extractors, filenames)
    num_rows = X.shape[0]
    is_vehicle = int('non-vehicles' not in directory)
    y = np.repeat(is_vehicle, num_rows)
    split_point = int(split*num_rows)
    return X[:split_point], X[split_point:], y[:split_point], y[split_point:]


bin_spatial_size = (16, 16)
color_hist_bins = 32
hog_orient = 8
hog_pix_per_cell = 16
hog_cell_per_block = 1
hog_channels = 'ALL'

feature_extractors = [
    # Take all channels from RGB
    lambda img: color_hist(img, nbins=color_hist_bins)[3],
    # Take only the S channel from HSV
    lambda img: color_hist(img, nbins=color_hist_bins, convert=cv2.COLOR_RGB2HSV)[1],
    # Take all channels from LUV
    lambda img: color_hist(img, nbins=color_hist_bins, convert=cv2.COLOR_RGB2LUV)[3],
    
    lambda img: bin_spatial(img, color_space='RGB', size=bin_spatial_size),
    lambda img: bin_spatial(img, color_space='HSV', size=bin_spatial_size),
    lambda img: bin_spatial(img, color_space='LUV', size=bin_spatial_size),
    
#     lambda img: get_hog_features(img, hog_orient, hog_pix_per_cell, hog_cell_per_block,
#                                  channel=hog_channels, color_space='RGB'),
    lambda img: get_hog_features(img, hog_orient, hog_pix_per_cell, hog_cell_per_block,
                                 channel=hog_channels, color_space='LUV'),
    lambda img: get_hog_features(img, hog_orient, hog_pix_per_cell, hog_cell_per_block,
                                 channel=hog_channels, color_space='HSV'),
]

X_train, X_test, y_train, y_test = create_dataset(
    feature_extractors,
    glob.glob('data/*/*'),
    split=0.8,
)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.utils import shuffle


svm_model = Pipeline([
    ('Scaler', StandardScaler()),
    ('SVM', LinearSVC()),
])

xgb_model = Pipeline([
    ('XGB', XGBClassifier(
        n_estimators=100,
        subsample=0.7,
        colsample_bytree=0.7,
    )),
])


xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred>0.5)
print('XGBoostClassifier: AUC: {:.2f}, Acc: {:.2f}'.format(auc, acc))

svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('LinearSVM: AUC: ----, Acc: {:.2f}'.format(acc))

In [ ]:
X = np.vstack([X_train, X_test])
y = np.hstack([y_train, y_test])

X, y = shuffle(X, y)

In [ ]:
%time xgb_model.fit(X, y)
%time _ = xgb_model.predict_proba(X)
plt.hist(_[y==1,1], bins=20);
plt.hist(_[y==0,1], bins=20);

In [ ]:
%time svm_model.fit(X, y)
%time _ = svm_model.predict(X)
plt.hist(_[y==1], bins=20);
plt.hist(_[y==0], bins=20);

# Sliding window

In [ ]:
def draw_boxes(img, bboxes, color, thick=6):
    draw_img = np.copy(img)
    for left_corner, right_corner in bboxes:
        cv2.rectangle(draw_img, left_corner, right_corner, color, thick)
    return draw_img
    

def slide_window(img, x_start_stop=[None, None], y_start_stop=[None, None], 
                 xy_window=(64, 64), xy_overlap=(0.5, 0.5)):
    if x_start_stop[0] is None:
        x_start_stop = (0, x_start_stop[1])
    if x_start_stop[1] is None:
        x_start_stop = (x_start_stop[0], img.shape[1])
    if y_start_stop[0] is None:
        y_start_stop = (0, y_start_stop[1])
    if y_start_stop[1] is None:
        y_start_stop = (y_start_stop[0], img.shape[0])
    # Compute the span of the region to be searched    
    xspan = x_start_stop[1] - x_start_stop[0]
    yspan = y_start_stop[1] - y_start_stop[0]
    # Compute the number of pixels per step in x/y
    nx_pix_per_step = np.int(xy_window[0]*(1 - xy_overlap[0]))
    ny_pix_per_step = np.int(xy_window[1]*(1 - xy_overlap[1]))
    # Compute the number of windows in x/y
    nx_buffer = np.int(xy_window[0]*(xy_overlap[0]))
    ny_buffer = np.int(xy_window[1]*(xy_overlap[1]))
    nx_windows = np.int((xspan-nx_buffer)/nx_pix_per_step) 
    ny_windows = np.int((yspan-ny_buffer)/ny_pix_per_step) 
    # Initialize a list to append window positions to
    window_list = []
    # Loop through finding x and y window positions
    # Note: you could vectorize this step, but in practice
    # you'll be considering windows one by one with your
    # classifier, so looping makes sense
    for ys in range(ny_windows):
        for xs in range(nx_windows):
            # Calculate window position
            startx = xs*nx_pix_per_step + x_start_stop[0]
            endx = startx + xy_window[0]
            starty = ys*ny_pix_per_step + y_start_stop[0]
            endy = starty + xy_window[1]
            # Append window position to list
            window_list.append(((startx, starty), (endx, endy)))
    # Return the list of windows
    return window_list


img = plt.imread('test_images/test1.jpg')

windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                       xy_window=(100, 100), xy_overlap=(0.5, 0.5))
                       
window_img = draw_boxes(np.copy(img), windows, color=(255., 255., 255.), thick=6)                    
plt.imshow(window_img)

In [ ]:
def search_windows(img, windows, clf, feature_extractors):
    on_windows = []
    
    probas_img = np.zeros_like(img).astype(np.float32)
    predictions_img = np.zeros_like(img).astype(np.float32)
    for window in windows:
        x_slc = slice(window[0][1], window[1][1])
        y_slc = slice(window[0][0], window[1][0])
        test_img = cv2.resize(img[x_slc, y_slc], (64, 64))
        test_img *= 255
        
        features = extract_features(feature_extractors, test_img)
        
        if hasattr(clf, 'predict_proba'):
            probas = clf.predict_proba(np.array([features]))[0, 1]
            prediction = (probas > 0.5)
            probas_img[x_slc, y_slc] += probas
        else:
            prediction = clf.predict(np.array([features]))
        
        if prediction == 1:
            predictions_img[x_slc, y_slc] += 1
            on_windows.append(window)
                            
    return on_windows, predictions_img, probas_img


def search_windows_faster(img, windows, clf, feature_extractors):
    on_windows = []
    
    probas_img = np.zeros_like(img).astype(np.float32)
    predictions_img = np.zeros_like(img).astype(np.float32)
    for window in windows:
        x_slc = slice(window[0][1], window[1][1])
        y_slc = slice(window[0][0], window[1][0])
        test_img = cv2.resize(img[x_slc, y_slc], (64, 64))
        test_img *= 255
        
        features = extract_features(feature_extractors, test_img)
        
        if hasattr(clf, 'predict_proba'):
            probas = clf.predict_proba(np.array([features]))[0, 1]
            prediction = (probas > 0.5)
            probas_img[x_slc, y_slc] += probas
        else:
            prediction = clf.predict(np.array([features]))
        
        if prediction == 1:
            predictions_img[x_slc, y_slc] += 1
            on_windows.append(window)
                            
    return on_windows, predictions_img, probas_img


model = xgb_model

for filename in sorted(glob.glob('test_images/*')):
    print(filename)
    img = plt.imread(filename)

    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 650), 
                           xy_window=(64, 64), xy_overlap=(0.5, 0.5))
    
    %time hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    window_img = draw_boxes(np.copy(img), hot_windows, color=(255, 0, 0), thick=4)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20,10))
    ax1.imshow(window_img)
    ax1.set_title('Original image', fontsize=30)
    ax2.imshow(predictions_img)
    ax2.set_title('Predictions', fontsize=30)
    ax3.imshow(probas_img)
    ax3.set_title('Probabilities', fontsize=30)
    prefix = filename.split('.')[0].split('/')[1]
    fig.savefig('output_images/' + prefix + '_out.jpg')
    
    plt.show()

In [ ]:
from scipy.ndimage.measurements import label


def draw_labeled_bboxes(img, labels):
    for car_number in range(1, labels[1]+1):
        nonzero = (labels[0] == car_number).nonzero()
        nonzeroy = np.array(nonzero[0])
        nonzerox = np.array(nonzero[1])    
        bbox = ((np.min(nonzerox), np.min(nonzeroy)), (np.max(nonzerox), np.max(nonzeroy)))    
        img = cv2.rectangle(img, bbox[0], bbox[1], (0, 0, 255), 6)
    return img

In [ ]:
model = xgb_model
overlap = 0.75

for filename in sorted(glob.glob('test_images/*')):
    img = plt.imread(filename)
    heatmap = np.zeros_like(img).astype(np.float32)
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(128, 128), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(86, 86), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(64, 64), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(48, 48), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(32, 32), xy_overlap=(0.25, 0.25))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    heatmap /= 5
    
    heatmap[heatmap <= 0.5] = 0
    
    labels = label(heatmap)
        
    window_img = draw_labeled_bboxes(img, labels)
    
    plt.imshow(window_img)
    prefix = filename.split('.')[0].split('/')[1]
    plt.savefig('output_images/' + prefix + '_boxes_out.jpg')
    plt.show()

In [ ]:
class PredictionStorage:
    def __init__(self, prev_count=1):
        self.prev_count = prev_count
        self.prev_heatmaps = []
        
    def get_agg_heatmap(self):
        return np.mean(self.prev_heatmaps, axis=0)
    
    def append_to_heatmap(self, heatmap):
        self.prev_heatmaps.append(heatmap)
        if len(self.prev_heatmaps) > self.prev_count:
            self.prev_heatmaps.pop(0)
        

def process_image(img, key_frame_interval=200, cache_length=100):
    global xgb_model, prediction_storage, overlap
    
    heatmap = np.zeros_like(img).astype(np.float32)
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(128, 128), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(86, 86), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(64, 64), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(48, 48), xy_overlap=(overlap, overlap))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    windows = slide_window(img, x_start_stop=(720, None), y_start_stop=(400, 550), 
                           xy_window=(32, 32), xy_overlap=(0.5, 0.5))
    hot_windows, predictions_img, probas_img = search_windows(img, windows, model, feature_extractors)
    heatmap += probas_img
    
    heatmap /= 5
    prediction_storage.append_to_heatmap(heatmap)
    heatmap = prediction_storage.get_agg_heatmap()
    
    heatmap[heatmap <= 0.5] = 0
    
    labels = label(heatmap)
    
    window_img = draw_labeled_bboxes(img, labels)
    
    return window_img

In [ ]:
from moviepy.editor import VideoFileClip




prediction_storage = PredictionStorage()

vid_output = 'project_video__mean5_out.mp4'
clip = VideoFileClip('project_video.mp4')

vid_clip = clip.fl_image(process_image)
vid_clip.write_videofile(vid_output, audio=False, threads=1)